In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "browser"

In [2]:
# Load a specific sheet
results = pd.read_excel('Concept testing.xlsx', sheet_name='Python')

overall_results = results[results['Delay'] == 'Overall']

minutes = [3,4,5,6,7,8,9,10,11,12,13,14,15,20,25]
minute_results = {}

for m in minutes:
    minute_results[m] = results[results['Delay'] == f'{m} minute']

In [3]:
for m in minutes: 

    df = minute_results[m].copy()

    df['Cat1'] = df[['KPI1','KPI2','KPI6','KPI8']].mean(axis=1)
    df['Cat2'] = df[['KPI3','KPI4','KPI5','KPI7','KPI8']].mean(axis=1)


    def pareto_front_max(df, x_col, y_col):
        points = df[[x_col, y_col]].values
        is_pareto = np.ones(points.shape[0], dtype=bool)

        for i, point in enumerate(points):
            if is_pareto[i]:
                dominated = np.any(
                    (points[:, 0] >= point[0]) &
                    (points[:, 1] >= point[1]) &
                    ((points[:, 0] > point[0]) | (points[:, 1] > point[1]))
                )
                if dominated:
                    is_pareto[i] = False

        return df[is_pareto]


    pareto_df = pareto_front_max(df, 'Cat1', 'Cat2').sort_values('Cat1')


    fig = go.Figure()

    # All designs
    fig.add_trace(go.Scatter(
        x=df['Cat1'],
        y=df['Cat2'],
        mode='markers+text',
        text=df['Unnamed: 0'],
        textposition='top center',
        name='Designs ({m} min)',
        marker=dict(
            size=10,
            color='skyblue',
            opacity=0.7,
            line=dict(width=1, color='black')
        ),
        hovertemplate=(
            "<b>%{text}</b><br>"
            "Good for ICE: %{x:.2f}<br>"
            "Good for Domestic Services: %{y:.2f}"
            "<extra></extra>"
        )
    ))

    # Pareto front (smooth + circular)
    fig.add_trace(go.Scatter(
        x=pareto_df['Cat1'],
        y=pareto_df['Cat2'],
        mode='lines+markers',
        name='Pareto Front ({m} min)',
        line=dict(
            color='red',
            width=3,
            shape='spline',
            smoothing=1.3
        ),
        marker=dict(
            symbol='circle',
            size=14,
            color='red',
            line=dict(width=2, color='darkred')
        ),
        hoverinfo='skip'
    ))


    fig.update_layout(
        title=dict(
            text='Pareto Front Analysis – {m} Minute Delay',
            x=0.5,
            font=dict(size=16)
        ),
        xaxis=dict(
            title='Good for ICE',
            gridcolor='rgba(0,0,0,0.15)'
        ),
        yaxis=dict(
            title='Good for Domestic Services',
            gridcolor='rgba(0,0,0,0.15)'
        ),
        template='plotly_white',
        legend=dict(
            x=0.02,
            y=0.98,
            bgcolor='rgba(255,255,255,0.8)'
        ),
        width=900,
        height=650
    )

fig.show()

In [11]:
import numpy as np
import plotly.graph_objects as go

# Pareto front function (define ONCE)
def pareto_front_max(df, x_col, y_col):
    points = df[[x_col, y_col]].values
    is_pareto = np.ones(points.shape[0], dtype=bool)

    for i, point in enumerate(points):
        if is_pareto[i]:
            dominated = np.any(
                (points[:, 0] >= point[0]) &
                (points[:, 1] >= point[1]) &
                ((points[:, 0] > point[0]) | (points[:, 1] > point[1]))
            )
            if dominated:
                is_pareto[i] = False

    return df[is_pareto]


# --------------------------------------------------
# Loop over minutes
# --------------------------------------------------
for m in minutes:

    df = minute_results[m].copy()

    # Compute category scores
    df['Cat1'] = df[['KPI1','KPI2','KPI6','KPI8']].mean(axis=1)
    df['Cat2'] = df[['KPI3','KPI4','KPI5','KPI7','KPI8']].mean(axis=1)

    pareto_df = pareto_front_max(df, 'Cat1', 'Cat2').sort_values('Cat1')

    fig = go.Figure()

    # All designs
    fig.add_trace(go.Scatter(
        x=df['Cat1'],
        y=df['Cat2'],
        mode='markers+text',
        text=df['Unnamed: 0'],
        textposition='top center',
        name=f'Designs ({m} min)',
        marker=dict(
            size=10,
            color='skyblue',
            opacity=0.7,
            line=dict(width=1, color='black')
        ),
        hovertemplate=(
            "<b>%{text}</b><br>"
            "Good for ICE: %{x:.2f}<br>"
            "Good for Domestic Services: %{y:.2f}"
            "<extra></extra>"
        )
    ))

    # Pareto front
    fig.add_trace(go.Scatter(
        x=pareto_df['Cat1'],
        y=pareto_df['Cat2'],
        mode='lines+markers',
        name=f'Pareto Front ({m} min)',
        line=dict(
            color='red',
            width=3,
            shape='spline',
            smoothing=1.3
        ),
        marker=dict(
            symbol='circle',
            size=14,
            color='red',
            line=dict(width=2, color='darkred')
        ),
        hoverinfo='skip'
    ))

    # Layout
    fig.update_layout(
        title=dict(
            text=f'Pareto Front Analysis – {m} Minute Delay',
            x=0.5,
            font=dict(size=16)
        ),
        xaxis_title='Good for ICE',
        yaxis_title='Good for Domestic Services',
        template='plotly_white',
        width=900,
        height=650
    )

    fig.write_image(f"Pareto_graphs/pareto_{m}_minute.png")
    #fig.show()


In [ ]:
import numpy as np
import plotly.graph_objects as go

# --------------------------------------------------
# Pareto front function (define once)
# --------------------------------------------------
def pareto_front_max(df, x_col, y_col):
    points = df[[x_col, y_col]].values
    is_pareto = np.ones(points.shape[0], dtype=bool)

    for i, point in enumerate(points):
        if is_pareto[i]:
            dominated = np.any(
                (points[:, 0] >= point[0]) &
                (points[:, 1] >= point[1]) &
                ((points[:, 0] > point[0]) | (points[:, 1] > point[1]))
            )
            if dominated:
                is_pareto[i] = False

    return df[is_pareto]

# --------------------------------------------------
# Precompute data for all minutes
# --------------------------------------------------
frames = []

for m in minutes:
    df = minute_results[m].copy()

    # Category scores
    df['Cat1'] = df[['KPI1','KPI2','KPI6','KPI8']].mean(axis=1)
    df['Cat2'] = df[['KPI3','KPI4','KPI5','KPI7','KPI8']].mean(axis=1)

    pareto_df = pareto_front_max(df, 'Cat1', 'Cat2').sort_values('Cat1')

    frames.append(
        go.Frame(
            name=str(m),
            data=[
                # All designs
                go.Scatter(
                    x=df['Cat1'],
                    y=df['Cat2'],
                    mode='markers+text',
                    text=df['Unnamed: 0'],
                    textposition='top center',
                    marker=dict(
                        size=10,
                        color='skyblue',
                        opacity=0.7,
                        line=dict(width=1, color='black')
                    )
                ),
                # Pareto front
                go.Scatter(
                    x=pareto_df['Cat1'],
                    y=pareto_df['Cat2'],
                    mode='lines+markers',
                    line=dict(
                        color='red',
                        width=3,
                        shape='spline',
                        smoothing=1.3
                    ),
                    marker=dict(
                        symbol='circle',
                        size=14,
                        color='red',
                        line=dict(width=2, color='darkred')
                    )
                )
            ],
            layout=go.Layout(
                title_text=f'Pareto Front Analysis – {m} Minute Delay'
            )
        )
    )

# --------------------------------------------------
# Initial figure (first minute)
# --------------------------------------------------
init_df = minute_results[minutes[0]].copy()
init_df['Cat1'] = init_df[['KPI1','KPI2','KPI6','KPI8']].mean(axis=1)
init_df['Cat2'] = init_df[['KPI3','KPI4','KPI5','KPI7','KPI8']].mean(axis=1)
init_pareto = pareto_front_max(init_df, 'Cat1', 'Cat2').sort_values('Cat1')

fig = go.Figure(
    data=[
        go.Scatter(
            x=init_df['Cat1'],
            y=init_df['Cat2'],
            mode='markers+text',
            text=init_df['Unnamed: 0'],
            textposition='top center',
            name='Designs',
            marker=dict(
                size=10,
                color='skyblue',
                opacity=0.7,
                line=dict(width=1, color='black')
            )
        ),
        go.Scatter(
            x=init_pareto['Cat1'],
            y=init_pareto['Cat2'],
            mode='lines+markers',
            name='Pareto Front',
            line=dict(
                color='red',
                width=3,
                shape='spline',
                smoothing=1.3
            ),
            marker=dict(
                symbol='circle',
                size=14,
                color='red',
                line=dict(width=2, color='darkred')
            )
        )
    ],
    frames=frames
)

# --------------------------------------------------
# Slider + layout
# --------------------------------------------------
fig.update_layout(
    template='plotly_white',
    width=900,
    height=650,
    xaxis_title='Good for ICE',
    yaxis_title='Good for Domestic Services',
    title=f'Pareto Front Analysis – {minutes[0]} Minute Delay',
    sliders=[{
        "active": 0,
        "currentvalue": {"prefix": "Delay: "},
        "pad": {"t": 50},
        "steps": [
            {
                "method": "animate",
                "label": f"{m} min",
                "args": [
                    [str(m)],
                    {"mode": "immediate", "frame": {"duration": 600}, "transition": {"duration": 300}}
                ]
            }
            for m in minutes
        ]
    }]
)
fig.write_html("pareto_slider.html")
